# 14장. 반복되는 분석 흐름을 자동화하기

이 노트북은 `book/chapters/ch14_airflow_pipeline.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 반복되는 분석 작업을 **입력 확인 → 전처리 → 분석 → 시각화 → 보고서 생성 → 산출물 검증** 단계로 나누고, Python 스크립트와 Docker Compose 기반 Airflow DAG로 실행 순서를 관리하는 것입니다.

## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 장은 먼저 Airflow 없이 Python 스크립트만으로 전체 흐름을 검증합니다.
- 그다음 같은 흐름을 Docker Compose 기반 Airflow DAG로 실행합니다.
- Docker Desktop 설치 방법은 별도 블로그 글을 참고합니다.
- 이 노트북에서는 Docker가 설치되어 있고 `docker`, `docker compose` 명령이 동작한다고 가정합니다.
- Airflow 설치 자체보다 DAG, Task 의존성, 로그인, 로그 확인, 산출물 검증에 집중합니다.

## 1. Docker 설치는 별도 가이드 참고

Docker 설치 과정은 이 강의안에 길게 포함하지 않습니다. 아래 블로그 글을 참고해 Docker Desktop을 설치합니다.

- Docker 설치 가이드: https://blog.naver.com/dev-dog/224341211248

설치 후 터미널에서 아래 명령이 정상 실행되는지 확인합니다.

```bash
docker --version
docker compose version
docker run hello-world
```

Windows에서는 Docker Desktop이 WSL2 backend를 사용하는 경우가 많습니다. 수업에서는 Docker Desktop 설치가 완료된 상태에서 Airflow 실습을 진행합니다.

## 2. 자동화는 코드를 대신 쓰는 일이 아니다

자동화는 분석 코드를 없애는 것이 아니라, 잘 정리된 분석 코드를 정해진 순서로 실행하도록 만드는 일입니다. 전처리, 분석, 시각화, 보고서 생성 코드가 뒤섞여 있으면 자동화하기 어렵습니다.

| 단계 | 하는 일 | 입력 | 출력 |
|---|---|---|---|
| 입력 확인 | 원본 데이터가 있는지 확인 | `data/raw/*.csv` | 확인 결과 |
| 전처리 | 결측치, 타입, 중복 처리 | 원본 CSV | `data/processed/*_clean.csv` |
| 분석 | 주요 지표 계산 | 전처리 데이터 | `reports/*.csv` |
| 시각화 | 그래프 생성 | 분석 결과 CSV | `reports/figures/*.png` |
| 보고서 | Markdown 보고서 작성 | 표, 그래프, 해석 문장 | `reports/*.md` |
| 검증 | 결과 파일 존재 여부 확인 | 산출물 목록 | 검증 로그 |

## 3. Make, n8n, Airflow의 역할 구분

| 도구 | 잘 맞는 상황 | 예시 |
|---|---|---|
| Make | 외부 서비스 연결과 알림 자동화 | 보고서 파일 생성 후 Gmail 발송, Slack 알림 |
| n8n | 노코드/로우코드 기반 워크플로우 구성 | API 호출, 데이터 저장, 내부 도구 연결 |
| Airflow | 코드 기반 데이터 파이프라인 운영 | 전처리 → 분석 → 시각화 → 보고서 생성 순서 관리 |

이 장에서는 Airflow가 분석 파이프라인을 실행하고, Make/n8n은 결과 전달 자동화에 활용할 수 있다는 관점으로 구분합니다.

## 4. 이번 장에서 완성할 파이프라인

이번 실습에서는 온라인 쇼핑몰 분석 흐름을 다음 순서로 자동화합니다.

```text
check_input_files
→ run_preprocessing
→ run_analysis
→ generate_visualizations
→ generate_report
→ validate_outputs
```

Airflow를 사용하기 전, 먼저 Python 함수와 스크립트로 이 순서가 정상 실행되는지 확인합니다.

## 5. 패키지와 경로 설정

프로젝트 루트, 원본 데이터 폴더, 전처리 데이터 폴더, 보고서 폴더, Docker Airflow 폴더를 설정합니다.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
FIGURE_DIR = REPORT_DIR / 'figures'
AIRFLOW_COMPOSE_DIR = PROJECT_ROOT / 'automation' / 'airflow'

for path in [RAW_DIR, PROCESSED_DIR, REPORT_DIR, FIGURE_DIR, AIRFLOW_COMPOSE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('프로젝트 루트:', PROJECT_ROOT)
print('Airflow Compose 폴더:', AIRFLOW_COMPOSE_DIR)


## 6. 파이프라인 Task 구조 확인하기

`src/automation_pipeline.py`에는 이번 장에서 사용할 공통 함수가 들어 있습니다. 먼저 Task 구조를 표로 확인합니다.

In [ ]:
from src.automation_pipeline import (
    check_input_files,
    create_airflow_setup_guide,
    create_pipeline_task_summary,
    generate_report,
    generate_visualizations,
    run_analysis,
    run_local_pipeline,
    run_preprocessing,
    validate_outputs,
)

task_summary = create_pipeline_task_summary()
task_summary.to_csv(REPORT_DIR / 'ch14_pipeline_task_summary.csv', index=False, encoding='utf-8-sig')
task_summary


## 7. Airflow 실행 전 Python 파이프라인 먼저 검증

Airflow에서 실패가 나면 Python 코드 문제인지, Docker/Airflow 환경 문제인지 구분하기 어렵습니다. 그래서 먼저 Airflow 없이 Python 파이프라인을 실행합니다.

터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/generate_sample_data.py
python scripts/run_ch14_pipeline.py
```

In [ ]:
input_check = check_input_files(PROJECT_ROOT)
input_check


In [ ]:
pipeline_result = run_local_pipeline(PROJECT_ROOT)
pipeline_result['validation_log']


## 8. Docker Compose Airflow 파일 구성

Docker 기반 Airflow 실습 파일은 `automation/airflow/` 폴더에 있습니다.

```text
automation/airflow/
├─ Dockerfile
├─ docker-compose.yml
├─ requirements.txt
├─ .env.example
└─ dags/
   └─ ch14_local_analysis_pipeline.py
```

`docker-compose.yml`은 프로젝트 루트를 컨테이너 내부의 `/opt/airflow/project`로 연결합니다. DAG에서는 이 경로를 기준으로 `scripts/ch14_*.py` 파일을 실행합니다.

In [ ]:
compose_files = [
    AIRFLOW_COMPOSE_DIR / 'Dockerfile',
    AIRFLOW_COMPOSE_DIR / 'docker-compose.yml',
    AIRFLOW_COMPOSE_DIR / 'requirements.txt',
    AIRFLOW_COMPOSE_DIR / '.env.example',
    AIRFLOW_COMPOSE_DIR / 'dags' / 'ch14_local_analysis_pipeline.py',
]

pd.DataFrame([
    {'file': str(path.relative_to(PROJECT_ROOT)), 'exists': path.exists()}
    for path in compose_files
])


## 9. Docker Compose로 Airflow 실행하기

터미널에서 아래 순서로 실행합니다. Windows PowerShell, macOS/Linux, WSL2 모두 같은 흐름을 사용할 수 있습니다.

```bash
cd automation/airflow
```

환경변수 예시 파일을 복사합니다.

macOS/Linux/WSL2:

```bash
cp .env.example .env
```

Windows PowerShell:

```powershell
copy .env.example .env
```

최초 1회 초기화:

```bash
docker compose up airflow-init
```

Airflow 실행:

```bash
docker compose up
```

## 10. Airflow UI 접속과 로그인

브라우저에서 아래 주소로 접속합니다.

```text
http://localhost:8080
```

아래와 같은 **Sign into Airflow** 화면이 나타나면 로그인 정보를 입력합니다.

![Airflow 로그인 화면](../images/airflow_login_screen.svg)

이미지가 보이지 않으면 [Airflow 로그인 화면 SVG 직접 보기](../images/airflow_login_screen.svg)를 클릭하세요.

이 레포지토리의 `automation/airflow/docker-compose.yml`은 수업용 기본 계정을 다음과 같이 생성하도록 설정되어 있습니다.

```text
ID: airflow
PW: airflow
```

따라서 14장 실습 환경에서는 먼저 `airflow / airflow`로 로그인합니다.

로그인 후 Airflow UI에서 `ch14_local_analysis_pipeline` DAG를 찾은 뒤 수동 실행합니다. Graph 또는 Grid 화면에서 Task 실행 순서와 실패 로그를 확인합니다.

## 10-1. `airflow / airflow`로 로그인이 안 될 때

로그인 화면에 다음 메시지가 보이면 기본 계정이 아니라 Simple Auth Manager의 자동 생성 비밀번호를 확인해야 할 수 있습니다.

```text
401 Unauthorized
Invalid credentials
Simple auth manager enabled
```

먼저 실행 중인 컨테이너 이름을 확인합니다.

```powershell
docker ps
```

14장 실습 환경에서는 보통 API 서버 컨테이너 이름이 `llm-course-airflow-api-server`입니다. 아래 명령으로 자동 생성 비밀번호 파일을 확인합니다.

```powershell
docker exec -it llm-course-airflow-api-server cat /opt/airflow/simple_auth_manager_passwords.json.generated
```

실제 실행 예시는 다음과 같습니다. 단, 아래 예시의 비밀번호는 보안상 마스킹했습니다. 실제 수업 환경에서는 오른쪽 문자열을 그대로 복사해 사용합니다.

```powershell
(.venv) PS D:\DEV\llm-data-analysis-course> docker exec -it llm-course-airflow-api-server cat /opt/airflow/simple_auth_manager_passwords.json.generated
{"admin": "생성된_비밀번호"}

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug llm-course-airflow-api-server
    Learn more at https://docs.docker.com/go/debug-cli/
(.venv) PS D:\DEV\llm-data-analysis-course>
```

위 출력에서 JSON의 왼쪽 값이 사용자 이름이고, 오른쪽 값이 비밀번호입니다.

```text
Username: admin
Password: 생성된_비밀번호
```

즉, 실제 출력이 다음과 비슷하다면:

```json
{"admin": "무작위_문자열"}
```

로그인 화면에는 다음처럼 입력합니다.

```text
Username: admin
Password: 무작위_문자열
```

Windows에서 로그에서 password 문자열을 검색하려면 다음 명령도 사용할 수 있습니다.

```powershell
docker logs llm-course-airflow-api-server | findstr /i password
```

macOS/Linux/WSL2에서는 다음처럼 확인합니다.

```bash
docker logs llm-course-airflow-api-server | grep -i password
```

`What's next: Try Docker Debug...` 문구는 Docker가 출력하는 안내 메시지입니다. Airflow 로그인 정보와 직접 관련이 없으므로 무시해도 됩니다.

## 10-2. 로그인 후 처음 열리는 화면에서 확인할 수 있는 것

로그인에 성공하면 아래와 같은 **Airflow 홈 대시보드**가 열립니다.

![Airflow 홈 대시보드](../images/airflow_home_dashboard.svg)

이미지가 보이지 않으면 [Airflow 홈 대시보드 SVG 직접 보기](../images/airflow_home_dashboard.svg)를 클릭하세요.

이 화면은 Airflow 전체 상태를 빠르게 확인하는 요약 화면입니다. 수업에서는 아래 항목을 먼저 보면 됩니다.

1. **실패한 Dags / 실행 중인 Dags / 활성 Dags**
   - 현재 실패한 DAG가 있는지
   - 지금 실행 중인 DAG가 있는지
   - 활성화된 DAG가 몇 개인지
   를 빠르게 확인할 수 있습니다.

2. **상태(Status)**
   - `메타데이터베이스`
   - `스케줄러`
   - `트리거러`
   - `Dag 프로세서`

   이 항목이 초록색이면 Airflow 핵심 구성요소가 정상 동작 중이라는 뜻입니다.

3. **풀 슬롯(Pool Slots)**
   - 현재 실행 자원과 슬롯 상태를 보여 줍니다.
   - 수업에서는 값 자체를 깊게 다루기보다, 전체가 정상으로 보이는지 정도만 확인하면 충분합니다.

4. **기록(Records) 영역의 Dag 실행들**
   - `대기 중`, `실행 중`, `성공`, `실패` 실행 수를 확인할 수 있습니다.
   - 아직 DAG를 실행하지 않았다면 대부분 `0`으로 보일 수 있습니다.

5. **에셋 이벤트들 / 태스크 인스턴스**
   - 에셋 기반 이벤트나 태스크 관련 요약이 표시됩니다.
   - 초보자는 이 영역보다 먼저 DAG 목록과 실행 결과 확인에 집중하면 됩니다.

6. **왼쪽 메뉴**
   - `Home`: 현재 홈 대시보드
   - `Dags`: DAG 목록 화면
   - `예셋`, `탐색`, `관리자`: 추가 관리 메뉴

수업에서 실제로 가장 자주 클릭하는 메뉴는 **왼쪽의 `Dags`** 입니다. 홈 화면에서 전체 상태를 확인한 뒤, `Dags` 메뉴로 이동해 `ch14_local_analysis_pipeline` DAG를 찾고 실행합니다.

즉, 로그인 후 첫 화면은 **“Airflow가 정상인지 확인하는 요약 화면”**, 실제 실습은 주로 **`Dags` 메뉴에서 진행**한다고 이해하면 됩니다.

## 10-3. Dags 목록에서 14장 실습 DAG 확인

왼쪽 메뉴에서 **Dags**를 클릭하면 아래와 같은 DAG 목록 화면이 열립니다.

![Airflow Dags 목록 화면](../images/airflow_dags_list_screen.svg)

이미지가 보이지 않으면 [Airflow Dags 목록 SVG 직접 보기](../images/airflow_dags_list_screen.svg)를 클릭하세요.

이 화면에서 `ch14_local_analysis_pipeline` DAG를 찾습니다. `sales_analysis_practice`와 같은 다른 DAG가 함께 보일 수 있으므로, 14장 실습에서는 반드시 `ch14_local_analysis_pipeline`을 클릭합니다.

DAG가 일시 중지 상태이면 토글을 켠 뒤 수동 실행합니다. 이후 Grid 또는 Graph 화면에서 Task 실행 순서와 실패 로그를 확인합니다.

## 10-4. DAG 실행 성공 화면 확인

`ch14_local_analysis_pipeline`을 수동 실행한 뒤 최근 실행이 초록색 체크로 표시되면 DAG 실행이 성공한 것입니다. 오른쪽의 Task 막대가 모두 초록색이면 `check_input_files`부터 `validate_outputs`까지 전체 Task가 정상 완료된 상태입니다.

![Airflow DAG 실행 성공 화면](../images/airflow_dag_success_screen.svg)

이미지가 보이지 않으면 [Airflow DAG 실행 성공 SVG 직접 보기](../images/airflow_dag_success_screen.svg)를 클릭하세요.

성공 화면에서 확인할 핵심은 다음 세 가지입니다.

1. `최근 실행` 시간 옆에 초록색 체크가 표시되는지 확인합니다.
2. 오른쪽 Task 막대가 모두 초록색인지 확인합니다.
3. 이후 `reports/ch14_airflow_validation_log.csv`에서 모든 `status`가 `ok`인지 확인합니다.

In [ ]:
# 참고용 명령어입니다. 실제 실행은 PowerShell 또는 터미널에서 진행하세요.
# docker exec -it llm-course-airflow-api-server cat /opt/airflow/simple_auth_manager_passwords.json.generated
# docker logs llm-course-airflow-api-server | findstr /i password


## 11. 산출물 검증

DAG가 정상 실행되면 프로젝트 루트의 `reports/` 폴더에 결과가 생성됩니다.

```text
reports/ch14_daily_sales.csv
reports/ch14_category_sales.csv
reports/figures/ch14_daily_sales.png
reports/ch14_airflow_report.md
reports/ch14_airflow_validation_log.csv
```

검증 로그에서 모든 `status`가 `ok`이면 주요 산출물이 정상 생성된 것입니다.

In [ ]:
validation_path = REPORT_DIR / 'ch14_airflow_validation_log.csv'
if validation_path.exists():
    display(pd.read_csv(validation_path))
else:
    print('아직 검증 로그가 없습니다. Python 파이프라인 또는 Airflow DAG를 먼저 실행하세요.')


## 12. 종료와 초기화

Airflow 컨테이너를 종료하려면 `automation/airflow` 폴더에서 다음 명령을 실행합니다.

```bash
docker compose down
```

Airflow DB 볼륨까지 완전히 삭제하려면 다음 명령을 사용합니다.

```bash
docker compose down --volumes --remove-orphans
```

이 명령은 DAG 실행 기록과 계정 정보도 초기화하므로 주의하세요.

## 13. 자주 발생하는 문제

### 8080 포트 충돌

이미 다른 프로그램이 8080 포트를 사용 중이면 `automation/airflow/docker-compose.yml`에서 포트를 바꿉니다.

```yaml
ports:
  - "8081:8080"
```

이후 `http://localhost:8081`로 접속합니다.

### 로그인 화면은 뜨지만 로그인이 안 됨

먼저 `airflow / airflow`를 입력합니다. 그래도 `401 Unauthorized` 또는 `Invalid credentials`가 나오면 Simple Auth Manager 비밀번호 파일을 확인합니다.

```powershell
docker exec -it llm-course-airflow-api-server cat /opt/airflow/simple_auth_manager_passwords.json.generated
```

출력 예시:

```json
{"admin": "생성된_비밀번호"}
```

이 경우 로그인 정보는 다음입니다.

```text
Username: admin
Password: 생성된_비밀번호
```

계정 정보를 완전히 초기화하려면 다음 명령을 사용할 수 있습니다. 단, DAG 실행 기록과 DB 볼륨도 함께 삭제됩니다.

```bash
docker compose down --volumes --remove-orphans
docker compose up airflow-init
docker compose up
```

### Docker 메모리 부족

Airflow는 여러 컨테이너를 실행하므로 Docker Desktop 메모리를 4GB 이상, 가능하면 8GB 정도로 설정하는 것을 권장합니다.

### ModuleNotFoundError

필요 패키지를 `automation/airflow/requirements.txt`에 추가하고 다시 빌드합니다.

```bash
docker compose build --no-cache
docker compose up airflow-init
docker compose up
```

## 14. Make와 n8n은 전달과 연결에 강하다

Airflow가 코드 기반 분석 파이프라인을 담당한다면, Make와 n8n은 결과물을 외부 서비스와 연결하는 데 유용합니다.

| 구간 | 담당 도구 예시 | 역할 |
|---|---|---|
| 데이터 처리 | Python, Airflow | 전처리, 분석, 시각화, 보고서 생성 |
| 결과 검증 | Python, Airflow | 파일 생성 여부, 크기, 로그 확인 |
| 외부 전달 | Make, n8n | 메일 발송, Slack 알림, Drive 업로드 |
| 운영 확인 | Airflow UI, Make/n8n 실행 로그 | 실패 지점과 재실행 여부 확인 |

## 15. LLM에게 파이프라인 설계를 요청하는 프롬프트

```text
온라인 쇼핑몰 주문 데이터를 매일 분석하는 자동화 파이프라인을 설계하려고 합니다.

실행 환경:
- Docker Desktop 설치 완료
- Docker Compose 기반 Airflow 실행
- 프로젝트 루트는 컨테이너 내부에서 /opt/airflow/project로 마운트됨

필요한 작업:
- 입력 파일 확인
- 데이터 전처리
- 매출 분석
- 시각화 생성
- Markdown 보고서 생성
- 결과 파일 검증
- 보고서 발송 또는 Slack 알림

요청:
1. 전체 작업을 단계별 Task로 나누어 주세요.
2. 각 Task의 입력과 출력을 표로 정리해 주세요.
3. Airflow가 담당할 부분과 Make/n8n이 담당할 부분을 나누어 주세요.
4. 실패했을 때 확인해야 할 로그와 검증 항목을 제안해 주세요.
5. Docker Compose 환경에서 경로 문제가 생기지 않도록 주의할 점을 알려 주세요.
```

## 16. 실습 과제

1. 별도 블로그 글을 참고해 Docker Desktop을 설치하고 `docker run hello-world`가 정상 실행되는지 확인하세요.
2. `python scripts/generate_sample_data.py`와 `python scripts/run_ch14_pipeline.py`를 실행해 Python 파이프라인을 검증하세요.
3. `automation/airflow` 폴더에서 `.env.example`을 `.env`로 복사하세요.
4. `docker compose up airflow-init`으로 Airflow DB를 초기화하세요.
5. `docker compose up`으로 Airflow를 실행하고 `http://localhost:8080`에 접속하세요.
6. 로그인 화면에서 먼저 `airflow / airflow`로 로그인하세요.
7. 로그인이 안 되면 `docker exec -it llm-course-airflow-api-server cat /opt/airflow/simple_auth_manager_passwords.json.generated`로 생성 비밀번호를 확인하고 `admin / 생성된_비밀번호`로 로그인하세요.
8. 로그인 후 홈 대시보드에서 실패한 Dags, 실행 중인 Dags, 상태(Status) 영역이 무엇을 의미하는지 확인하세요.
9. `Dags` 메뉴로 이동해 `ch14_local_analysis_pipeline` DAG를 수동 실행하고 Task 실행 순서를 확인하세요.
10. 최근 실행 옆 초록색 체크와 Task 막대가 모두 초록색인지 확인하세요.
11. `reports/ch14_airflow_validation_log.csv`에서 모든 `status`가 `ok`인지 확인하세요.
12. 입력 파일 하나를 임시로 바꿔 실패 상황을 만들고 Airflow 로그를 확인하세요.

In [ ]:
# 과제: 생성된 14장 산출물 목록을 확인해 보세요.
for path in sorted(REPORT_DIR.glob('ch14_*')):
    print(path.name, path.stat().st_size if path.exists() else 'missing')


## 17. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 반복 분석 업무를 Task로 나누는 방법
- Make, n8n, Airflow의 역할 구분
- Docker 설치는 별도 가이드로 분리하고, 강의안은 Docker Compose 실행에 집중하는 방식
- Airflow 실행 전 Python 스크립트로 분석 코드 사전 검증
- Docker Compose 기반 Airflow 실행 구조
- `automation/airflow/docker-compose.yml`, `Dockerfile`, `requirements.txt`, `.env.example` 구성
- Airflow 로그인 화면에서 `airflow / airflow`와 Simple Auth Manager 생성 비밀번호 확인 방법
- 로그인 후 홈 대시보드에서 DAG 현황, 상태, 실행 기록을 읽는 방법
- Airflow Dags 목록에서 `ch14_local_analysis_pipeline` DAG를 확인하는 방법
- Airflow DAG 실행 성공 화면에서 최근 실행 초록색 체크와 Task 막대를 확인하는 방법
- Airflow UI에서 DAG 실행, 실패 Task 로그 확인, 산출물 검증
- 자동화 결과의 실행 성공, 산출물 성공, 분석 품질 구분

다음 장에서는 지금까지 배운 EDA, 시각화, 머신러닝, LLM 활용, 외부 데이터, 자동화 아이디어를 기말 프로젝트로 통합합니다.